In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Evaluate Managed Agents

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/evaluate_managed_agents.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fevaluation%2Fevaluate_managed_agents.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/evaluation/evaluate_managed_agents.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/evaluate_managed_agents.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

| Authors |
| --- |
| [Kelsi Lakey](https://github.com/lakeyk) |
| [Jason Dai](https://github.com/jsndai) |

## Overview

This notebook demonstrates how to evaluate agents built with the
[Managed Agents API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/managed-agents)
using the Gen AI Evaluation Service.

You will learn how to:

- **Generate conversation scenarios** from your agent's configuration
- **Run inference** to execute your agent against the generated scenarios
- **Evaluate** the resulting conversation traces with prebuilt metrics
- **Evaluate existing interactions** recorded with the Interactions API

## Get started

### Install Google Gen AI SDK and other required packages

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform[evaluation]

### Authenticate your notebook environment

If you are running this notebook in **Google Colab**, run the cell below to
authenticate your account.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

To get started using Agent Platform, you must have an existing Google Cloud
project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects)
and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [ ]:
import os

import agentplatform
from agentplatform import types as ap_types

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
LOCATION = "global"  # @param {type: "string"}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

client = agentplatform.Client(project=PROJECT_ID, location=LOCATION)

### Set agent resource

Set the agent resource name for the Managed Agent you want to evaluate.
To create a Managed Agent, see
[Create and manage agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/managed-agents/create-manage).

In [ ]:
AGENT_ID = "your-agent-id"  # @param {type: "string"}

AGENT_RESOURCE = f"projects/{PROJECT_ID}/locations/{LOCATION}/agents/{AGENT_ID}"

## Generate conversation scenarios

Use `generate_conversation_scenarios` to automatically create diverse,
multi-turn test scenarios from your agent's system instruction and tool
definitions. This lets you start testing immediately without manually
authoring test cases.

In [ ]:
SCENARIO_COUNT = 5  # @param {type: "integer"}
GENERATION_INSTRUCTION = "Create agent scenarios"  # @param {type:"string"}

scenarios = client.evals.generate_conversation_scenarios(
    agent=AGENT_RESOURCE,
    config={
        "count": SCENARIO_COUNT,
        "generation_instruction": GENERATION_INSTRUCTION,
    },
)

scenarios.show()

## Run inference

Use `run_inference` to execute your agent against the generated scenarios.
Each scenario produces a conversation trace that captures the agent's
full behavior, including tool calls, intermediate steps, and the final
response.

In [ ]:
MAX_TURNS = 3  # @param {type: "integer"}

inference_results = client.evals.run_inference(
    agent=AGENT_RESOURCE,
    src=scenarios,
    config={"user_simulator_config": {"max_turn": MAX_TURNS}},
)

inference_results.show()

## Evaluate

Use `evaluate` to score the conversation traces with prebuilt metrics.

In [ ]:
eval_result = client.evals.evaluate(
    dataset=inference_results,
    metrics=[
        ap_types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
    ],
    agent=AGENT_RESOURCE,
)

eval_result.show()

## Evaluate existing interactions

You can also evaluate interactions that have already been recorded with your
agent. This is useful for assessing the quality of production conversations.
You can create interactions by following
[Interact with agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/managed-agents/interact-with-agents).

In [ ]:
INTERACTION_ID = "your-interaction-id"  # @param {type: "string"}

interactions_dataset = ap_types.EvaluationDataset(
    eval_cases=[
        ap_types.EvalCase(
            interactions_data_source=ap_types.InteractionsDataSource(
                interaction=(
                    f"projects/{PROJECT_ID}/locations/{LOCATION}"
                    f"/interactions/{INTERACTION_ID}"
                ),
                gemini_agent_config=ap_types.GeminiAgentConfig(
                    gemini_agent=AGENT_RESOURCE,
                ),
            ),
        ),
    ]
)

eval_result = client.evals.evaluate(
    dataset=interactions_dataset,
    metrics=[
        ap_types.RubricMetric.MULTI_TURN_TASK_SUCCESS,
    ],
    agent=AGENT_RESOURCE,
)

eval_result.show()

## Cleaning up

To clean up all Google Cloud resources used in this project, you can
[delete the Google Cloud project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects)
you used for the tutorial.

Otherwise, you can delete the individual agent resource:

In [ ]:
# Delete the agent
# client.agents.delete(name=AGENT_RESOURCE)